# 02 — Les bras du factoriel et l'équité

**Notebook de cours — `structured-latent-dynamics`**

Le notebook 00 a établi QUE la structure coûte sur le banc D3, à un niveau `off`/`on` qui
restait volontairement vague sur ce que « les bras » recouvrent. Le notebook 01 a construit
le bloc $2\times2$ lui-même — sa filiation neuronale, son rayon spectral, la reparamétrisation
qui le borne. Ce notebook répond à la question que les deux précédents laissent ouverte :
**avec quoi entoure-t-on ce bloc pour produire un contexte $c_t$, et à quelle condition la
comparaison entre ces choix est-elle honnête ?**

Trois objets nouveaux : les producteurs de contexte du facteur `structure` (dont l'un, le
**minimal causal transformer**, n'a encore jamais été expliqué dans cette série) ; le wrapper
complet $\psi/\bar A/\pi$ qui les enveloppe tous, avec sa dégénérescence de jauge ; et le
contrat de budget à $\pm5\%$ qui rend une comparaison entre bras autre chose qu'une
comparaison de largeur de réseau déguisée en comparaison de structure. Ce dernier point n'est
pas une précaution abstraite — c'est la leçon la plus coûteuse que ce dépôt ait payée, et la
§1 raconte pourquoi.

Ce notebook ne fait tourner aucun entraînement : ce qu'il prouve, il le prouve par dérivation,
par implémentation minimale et par parité numérique contre le code réel de `sld`.

## Plan et conventions

| § | Contenu |
|---|---|
| 1 | Pourquoi comparer à bras équitables — la leçon de la campagne μ_J |
| 2 | Les backbones du facteur `structure`, et le nom hérité qui n'en fait pas partie |
| 3 | Le wrapper $\psi/\bar A/\pi$ : la cellule, le théorème de jauge, le correctif partiel |
| 4 | Le budget à $\pm5\%$ : `match_budget`, la suspension par nom, les comptes recalculés |
| 5 | Le piège : trois refus que le code oppose à une mauvaise configuration |
| 6 | Conclusion et pont vers le notebook 03 |

**Convention de citation**, reprise sans changement de la série : un chiffre **[dépôt]** vient
d'un fichier nommé du dépôt, jamais reproduit à la volée par ce notebook ; un chiffre
**reproduit ici** est recalculé par une cellule de code qui suit, à chaque exécution.

**Sources citées dans ce notebook** : `PROTOCOL.md` §11 (retrait de P1(b)), §13 (le facteur
structure devient architectural), §28.2 (la racine `results_basis`) ; `REGLES.md` §3
(vocabulaire) et §4 (équité) ; `FINDINGS.md` §6 ; `docs/historical_mu_j_campaign.md` ;
`src/sld/predictors/{transformer,structured,blocks2x2,mlp,min_ssm,koopman_ssm,registry}.py` ;
`src/sld/training/budget.py` ; `src/sld/systems/coupled_oscillators.py` ;
`src/sld/experiments/configs/basis_root.yaml` ; `tests/test_basis_root.py`.

In [ ]:
from pathlib import Path
import math

import numpy as np
import torch
import yaml

np.set_printoptions(precision=6, suppress=False, linewidth=110)
torch.set_printoptions(precision=6, sci_mode=False)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

print("numpy", np.__version__, "| torch", torch.__version__)
print("REPO =", REPO)

## 1. Pourquoi comparer à bras équitables

Comparer `structure: off` à `structure: on` n'a de sens qu'à une condition : que les deux
bras diffèrent **exactement** par la présence de la transition structurée, et par rien
d'autre. Si le bras `on` est aussi, incidemment, un réseau plus large, un écart mesuré entre
les deux ne dit rien sur la structure — il dit quelque chose sur la largeur, habillé du nom
de la structure. C'est une faute de design, pas une négligence de reporting, et ce dépôt l'a
payée une fois en entier avant de la corriger par contrat plutôt que par vigilance.

### 1.1 La leçon que la campagne μ_J a payée

Ce dépôt a d'abord tenté de rendre la comparaison équitable par un **niveau de contrainte
calibré** — une marge spectrale amenée dans une bande commune `[0.90, 1.00]` par un
coefficient de pénalité μ_J, un par famille. Trois familles sur quatre n'ont jamais atteint
cette bande, à quatre décades de μ_J. La prédiction qu'elle devait trancher — « à contrainte
fixée, la paramétrisation spectrale explicite bat les familles génériques » — a donc été
**retirée comme non identifiée**, et non falsifiée : il n'existe aucune mesure où la
comparaison a jamais eu lieu au niveau qu'elle annonçait comparer. Le détail de ce qui a été
testé, diagnostiqué et réfuté tient dans `docs/historical_mu_j_campaign.md` et `PROTOCOL.md`
§11 (Amendment 2, « withdrawal of P1(b) ») — cette campagne est tranchée et close, ce notebook
ne la rejoue pas.

Ce qui en reste, utile pour la suite : une bande calibrée sur un scalaire de stabilité mesuré
séparément par famille n'est pas un contrat d'équité opposable — elle se laisse discuter, et
elle *a* été discutée jusqu'à l'échec. Le contrat d'équité que ce dépôt retient depuis
(`REGLES.md` §4 : « params ±5%, FLOPs comptés, mêmes updates, même sweep de tuning par
bras ») compare les bras autrement : à **nombre de paramètres égal, à $\pm5\%$**, un contrat
mécanique, vérifiable par un compte d'entiers plutôt que par une calibration. La §4 de ce
notebook le construit et le vérifie sur les bras réels ; la §2 explique d'abord *quels* bras
il compare, depuis que `PROTOCOL.md` §13 (Amendment 4) a fait du backbone un axe
architectural.

## 2. Les bras du facteur `structure` : les backbones, et leurs deux rôles

Le facteur `structure` a deux niveaux, `off` et `on`, croisés avec un facteur `backbone` à
trois niveaux — `mlp`, `transformer`, `min_ssm`. Cela donne six cellules, pas neuf : chaque
backbone joue **deux rôles**, un pour chaque niveau de `structure`, et le code le reflète
littéralement — une classe autonome pour le rôle `off`, un producteur de contexte pour le
rôle `on` :

| backbone | rôle `off` (prédicteur autonome) | rôle `on` (producteur de contexte $\kappa_b$) |
|---|---|---|
| `mlp` | `MLPPredictor` | `MLPContextBackbone` → `mlp_k` |
| `transformer` | `CausalTransformerPredictor` | `CausalTransformerContextBackbone` → `transformer_k` |
| `min_ssm` | `SelectiveSSMPredictor` | `SelectiveSSMContextBackbone` → `ssm_k` |

C'est pourquoi la comparaison « `X` contre `X_k` », à backbone fixé, est celle que
`PROTOCOL.md` §13.7 déclare bien posée : les deux cellules partagent le même producteur de
contexte sous ses deux rôles, et ne diffèrent que par la présence du wrapper $\psi/\bar
A/\pi$ de la §3. La comparaison entre `mlp_k`, `transformer_k` et `ssm_k`, à `structure: on`
fixé, compare en revanche trois choix de backbone différents *à l'intérieur* du même wrapper
— à budget total égal mais, on le verra en §4, à capacité $\psi/\pi$ **inégale**.

**`mlp`/`mlp_k` — la baseline mémoire-less.** Le plus simple des trois : `MLPPredictor`
calcule `h_next = h + MLP(concat([h, u]))`, sans état récurrent (`PredictorState.m` reste
`None`) ; `MLPContextBackbone` produit `c_t = MLP(h_t)` de la même façon, sans jamais voir
`u_t`. C'est la colonne du factoriel sans mémoire, contre laquelle les deux backbones
suivants — qui, eux, portent un état — se lisent.

### 2.2 Le « minimal causal transformer » — pourquoi la causalité n'est pas un détail

Le deuxième backbone porte, dans ce dépôt, le nom honnête de **minimal causal transformer**
(`REGLES.md` §3 : on ne survend pas l'échelle — ce n'est ni GPT ni un transformer de
production, c'est le strict nécessaire pour tester si l'attention, en tant que mécanisme,
aide ou nuit à ce banc). Il apparaît sous deux formes dans `src/sld/predictors/transformer.py`
: `CausalTransformerPredictor` (le rôle `off`, colonne `transformer`) et
`CausalTransformerContextBackbone` (le rôle `on`, colonne `transformer_k`) — même mécanisme
d'attention, deux façons de le brancher.

**Pourquoi causal, et pourquoi ce n'est pas négociable.** La tâche que ce dépôt pose est la
prédiction du prochain état latent $h_{t+1}$ à partir de l'histoire $(h_{\le t}, u_{\le t})$
— jamais du futur. Un transformer standard laisse chaque position attendre **toutes** les
autres, passé et futur confondus ; ici, la position $t$ ne doit attendre que $s \le t$. Ce
n'est pas une optimisation, c'est une **contrainte de la tâche elle-même** : un modèle qui
violerait la causalité pourrait tricher à l'entraînement en lisant $h_{t+1}$ à travers
l'attention, et le rollout en ligne — où le futur n'existe simplement pas encore — révélerait
immédiatement la triche par un score d'entraînement sans rapport avec la performance réelle.

**Le masque.** Pour une séquence de $T$ tokens $x_1,\ldots,x_T\in\mathbb{R}^d$,
l'auto-attention à une tête calcule

$$\mathrm{scores}_{ij} = \frac{q_i\cdot k_j}{\sqrt d}, \qquad
\mathrm{attn}_i = \mathrm{softmax}_j\big(\mathrm{scores}_{ij} + \mathrm{mask}_{ij}\big), \qquad
\mathrm{out}_i = \sum_j \mathrm{attn}_{ij}\,v_j,$$

où $\mathrm{mask}_{ij} = -\infty$ si $j>i$ (le futur de $i$) et $0$ sinon. Le $-\infty$ envoie
le poids d'attention correspondant à exactement $0$ après le softmax — la position $i$ ne
peut littéralement pas lire $j>i$, pas seulement « avec un faible poids ».

**Le cache KV.** Recalculer $\mathrm{scores}_{ij}$ pour tout $j\le i$ à chaque nouveau pas $i$
referait tout le travail déjà fait aux pas précédents. Le cache conserve les clés et valeurs
déjà produites — `PredictorState.m` empaquette, pour `CausalTransformerPredictor`, un tenseur
de forme `(n_layers, 2, B, n_heads, t, d_head)` (l'indice 2 sépare K de V) — et chaque nouveau
pas ne calcule que **sa propre** requête, clé et valeur, qu'il concatène au cache existant
avant d'attendre dessus. La cellule qui suit démontre que cette économie n'est **pas une
approximation** : la sortie au pas $t$ obtenue par cache est identique, à la précision
machine, à celle obtenue en recalculant l'attention complète sur toute la séquence avec le
masque ci-dessus. C'est la raison pour laquelle un rollout de longueur $H$ coûte $O(H^2)$ au
total (chaque pas $t$ coûte $O(t)$, sommé sur $t=1..H$) plutôt que $O(H^3)$ — et c'est aussi
pourquoi cette famille est déclarée, dans le code même, comme « the generic
$O(H^2)$-per-trajectory baseline », jamais comme une architecture passant à l'échelle.

**Ce que « minimal » retire, précisément.** Deux blocs pré-LN (attention causale + MLP, deux
résidus), un encodage positionnel **sinusoïdal fixe** — non appris, donc sans paramètre
supplémentaire à faire entrer dans le budget de la §4, ce qui garde la comparaison de
paramètres entre familles honnête — et une tête de sortie résiduelle `h_next = h +
Linear(y_t)`, identique en forme à celle de `mlp`. Rien d'autre : pas de dropout, pas de
dizaines de couches, pas d'embedding de vocabulaire (l'entrée est déjà continue). Le rôle
`on` (`CausalTransformerContextBackbone`) ajoute une seule différence structurelle, imposée
par le contrat du wrapper (§3) : la requête au pas $t$ vient de $h_t$ **seul**, jamais de
$u_t$, tandis que le cache se remplit de paires $(h_s, u_s)$ pour $s<t$ — deux appels
séparés, `context_step` puis `absorb`, si bien que $u_t$ ne peut structurellement pas
atteindre $c_t$.

In [ ]:
# --- [DEMONTRE -> verifie] Le cache KV n'est pas une approximation du masque causal --------
rng = np.random.default_rng(0)
T, d = 5, 3
Q = rng.standard_normal((T, d))
K = rng.standard_normal((T, d))
V = rng.standard_normal((T, d))


def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)


# (a) un seul passage, masque causal T x T : la position i ignore j > i
scores = (Q @ K.T) / np.sqrt(d)
future = np.triu(np.ones((T, T), dtype=bool), k=1)
scores_masked = np.where(future, -np.inf, scores)
out_masked = softmax(scores_masked) @ V

# (b) cache KV qui grandit pas a pas : au pas t, la requete t n'attend que sur k <= t
out_cache = np.zeros_like(out_masked)
for t in range(T):
    k_t, v_t = K[: t + 1], V[: t + 1]
    s_t = (Q[t : t + 1] @ k_t.T) / np.sqrt(d)
    out_cache[t] = softmax(s_t) @ v_t

diff = np.abs(out_masked - out_cache).max()
print(f"|masque complet - cache incremental|_inf = {diff:.3e}")
assert np.allclose(out_masked, out_cache, atol=1e-12)

**Lecture.** Les deux calculs coïncident à l'arrondi machine près : faire grandir le
cache pas à pas donne, à chaque pas, exactement la même sortie que recalculer l'attention
complète sur toute la séquence vue jusque-là avec le masque causal. Le cache n'approxime
rien — il évite de refaire un calcul déjà fait. C'est cette identité, et elle seule, qui
justifie de parler d'un coût $O(t)$ par pas plutôt que $O(t^2)$.

In [ ]:
# --- [PARITE PACKAGE] CausalTransformerPredictor.step, cache reel, contre le calcul a la main
from sld.predictors.transformer import CausalTransformerPredictor
from sld.predictors.base import PredictorState

torch.manual_seed(0)
D_H, D_U_TF, D_MODEL = 2, 1, 4
model_tf = CausalTransformerPredictor(d_h=D_H, d_u=D_U_TF, d_model=D_MODEL, n_heads=1, n_layers=1)
model_tf.eval()

T = 4
rng_tf = np.random.default_rng(1)
h_seq = rng_tf.standard_normal((T, D_H)).astype(np.float32)
u_seq = rng_tf.standard_normal((T, D_U_TF)).astype(np.float32)

# (A) le package : cache KV reel, un pas a la fois, sur la MEME sequence externe (h_t, u_t)
state = PredictorState()
outs_cache = []
with torch.no_grad():
    for t in range(T):
        h_next, state = model_tf.step(
            torch.tensor(h_seq[t : t + 1]), torch.tensor(u_seq[t : t + 1]), state
        )
        outs_cache.append(h_next.numpy()[0])
outs_cache = np.stack(outs_cache)

# (B) a la main : un seul passage complet, masque causal explicite, poids extraits du module
sd = {k: v.detach().numpy() for k, v in model_tf.state_dict().items()}
pe = model_tf.pe.detach().numpy()


def linear(x, W, b):
    return x @ W.T + b


def layernorm(x, w, b, eps=1e-5):
    mu, var = x.mean(-1, keepdims=True), x.var(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps) * w + b


def gelu(x):
    return 0.5 * x * (1.0 + np.vectorize(math.erf)(x / math.sqrt(2.0)))


tokens = np.concatenate([h_seq, u_seq], axis=-1)
x = linear(tokens, sd["in_proj.weight"], sd["in_proj.bias"]) + pe[:T]

xn = layernorm(x, sd["blocks.0.ln1.weight"], sd["blocks.0.ln1.bias"])
q, k, v = np.split(linear(xn, sd["blocks.0.qkv.weight"], sd["blocks.0.qkv.bias"]), 3, axis=-1)
scores_tf = (q @ k.T) / math.sqrt(D_MODEL)
future_tf = np.triu(np.ones((T, T), dtype=bool), k=1)
attn_tf = softmax(np.where(future_tf, -np.inf, scores_tf))
x = x + linear(attn_tf @ v, sd["blocks.0.attn_out.weight"], sd["blocks.0.attn_out.bias"])
xn2 = layernorm(x, sd["blocks.0.ln2.weight"], sd["blocks.0.ln2.bias"])
hidden = gelu(linear(xn2, sd["blocks.0.mlp.0.weight"], sd["blocks.0.mlp.0.bias"]))
x = x + linear(hidden, sd["blocks.0.mlp.2.weight"], sd["blocks.0.mlp.2.bias"])
outs_scratch = h_seq + linear(x, sd["out_proj.weight"], sd["out_proj.bias"])

diff = np.abs(outs_cache - outs_scratch).max()
print(f"|cache incremental (package) - masque complet (a la main)|_inf = {diff:.3e}   (tolerance 1e-5, float32)")
assert diff < 1e-5

**Lecture.** Même identité que la cellule précédente, cette fois sur le vrai module
`sld`, avec les poids réels extraits par `state_dict()` (`pe` est un buffer non persistant,
donc lu séparément sur `model.pe`) — pas une reproduction indépendante approximative. L'écart
mesuré, de l'ordre de $10^{-7}$, est la précision `float32` accumulée sur une LayerNorm, une
softmax et deux couches linéaires, pas un défaut de méthode : la cellule précédente, en
`float64` sans couche apprise, atteignait $0$ exactement. Ceci clôt la démonstration : le
cache KV de `CausalTransformerPredictor.step` calcule, pas à pas, exactement ce que
calculerait un passage complet masqué — la seule différence est le budget de calcul, jamais
le résultat.

### 2.4 `min_ssm` / `ssm_k` — un backbone déjà rencontré, pas re-dérivé

Nom honnête (`REGLES.md` §3) : **Mamba-style minimal selective SSM** (Gu & Dao, 2023) — un
état récurrent `m` dont l'opérateur de transition est choisi par l'état courant lui-même
(`SelectiveSSMPredictor`, rôle `off` ; `SelectiveSSMContextBackbone`, rôle `on` → `ssm_k`,
`src/sld/predictors/min_ssm.py`). Son mode d'échec — norme de gradient au premier pas
$2.4\times10^{18}$ contre $28.2$ pour `mlp`, perte non finie dès l'époque 1 — a déjà été
enseigné, avec sa cause et son remède (le pas latent borné, `delta_max`), au notebook 01,
§3.2 ; il n'est pas redérivé ici.

### 2.5 `koopman_ssm` — un héritage, jamais une cellule du factoriel

Le septième nom du registre n'a pas de rôle `off`/`on` : `koopman_ssm`
(`KoopmanSSMPredictor`, `src/sld/predictors/koopman_ssm.py`) est ce que la cellule `(ssm,
on)` **désignait avant** l'amendement 4 (`PROTOCOL.md` §13.1, daté du 2026-08-04) —
`constrained=True`, `alpha = -exp(nu)`, contractif par construction. L'amendement a déplacé
l'axe `structure` sur les bras `_k` bâtis sur un seul wrapper partagé (§3), et `koopman_ssm`
est **retenu, pas réinterprété** : la classe reste constructible pour que les runs déjà sur
disque restent chargeables, mais elle est **refusée** comme cellule du factoriel avant tout
run (`registry.py`, `STRUCTURED_FAMILIES` ne la contient pas). Le registre porte donc sept
noms pour six cellules — une asymétrie déclarée, pas un oubli.

`koopman_ssm` couvre trois régimes de transition, et c'est là que la discipline de
vocabulaire de `REGLES.md` §3 se joue concrètement :

- **`"constant"`** — un opérateur discret **fixe** $\bar A$, entrées additives par $\bar
  B_0$ : régime Koopman (i), lecture EDMD (Williams, Kevrekidis & Rowley, 2015).
- **`"bilinear"`** — le régime (ii) : la même forme affine, **plus** le terme d'interaction
  explicite $\sum_j u_j(\bar B_j h)$ — c'est ce terme, et lui seul, qui autorise le mot
  « bilinéaire ».
- **`"selective"`** — sort du cadre Koopman : un pas $\delta_t = f(h,u)$ dépendant de l'état
  module le champ de blocs. L'opérateur devient **LPV/cocycle**, nommé comme tel dans le
  code, jamais Koopman.

Le wrapper de la §3, lui, produit un $\bar A_t$ qui dépend du contexte $c_t$ — donc
**LPV/cocycle** par construction, quel que soit le backbone qui produit $c_t$, et sa
transition $\bar A_t h_t + \bar B_t u_t$ reste **affine** en $(h_t,u_t)$ une fois $c_t$ figé,
jamais bilinéaire au sens ci-dessus : aucun bras `_k` ne porte le terme d'interaction
explicite. Aucun des bras `_k` n'est, à aucun titre, un opérateur Koopman.

In [ ]:
# --- [VERIFIE] Sept noms, six cellules : koopman_ssm est absent du factoriel -----------------
from sld.predictors.registry import STRUCTURED_FAMILIES, _FAMILY_CONSTRUCTORS

print("familles du registre :", sorted(_FAMILY_CONSTRUCTORS))
print("bras structures du factoriel :", STRUCTURED_FAMILIES)
assert len(_FAMILY_CONSTRUCTORS) == 7
assert len(STRUCTURED_FAMILIES) == 3
assert "koopman_ssm" in _FAMILY_CONSTRUCTORS and "koopman_ssm" not in STRUCTURED_FAMILIES
print(
    "koopman_ssm : constructible =", True,
    "| cellule du factoriel =", "koopman_ssm" in STRUCTURED_FAMILIES,
)

**Lecture.** Le compte confirme la lecture textuelle du code : sept classes sont
constructibles (`_FAMILY_CONSTRUCTORS`), trois seulement portent le suffixe `_k` du wrapper
structuré (`STRUCTURED_FAMILIES`), et `koopman_ssm` appartient au premier ensemble sans
appartenir au second. Ce n'est pas une convention de nommage :
`sld.experiments.factorial._validate_arm` refuse explicitement toute cellule qui nommerait
`koopman_ssm` avant qu'aucun run ne parte.

## 3. Le wrapper structuré : $\psi$, $\bar A$, $\pi$

Quel que soit le backbone choisi en §2, le bras `on` qui en résulte enveloppe le même
wrapper (`StructuredPredictor`, **`src/sld/predictors/structured.py`**) :

$$h_0 = \psi(z_0), \qquad c_t = \kappa_b(h_t, m_t), \qquad
h_{t+1} = \bar A_t\,h_t + \bar B_t\,u_t, \qquad \hat z_t = \pi(h_t),$$

où $\bar A_t$ est bloc-diagonale de blocs rotation-scaling (notebook 01), $\bar B_t =
\Phi_t B_t$ avec $\Phi_t$ l'intégrale ZOH, et où $\psi$ et $\pi$ sont, au niveau
`lift="mlp"` (le défaut), **deux MLP indépendants** — aucun terme de reconstruction dans les
objectifs, aucune contrainte d'inversibilité entre eux. Une fois $c_t$ figé, cette
transition est **affine** en $(h_t, u_t)$, jamais bilinéaire (§2.5) : aucun terme
d'interaction explicite $\sum_j u_j \bar B_j h$ n'y figure.

**Les trois niveaux de `lift`** (`_VALID_LIFT`, refusé sans repli si mal orthographié) :

| `lift` | $\psi$ / $\pi$ | ce que ça teste |
|---|---|---|
| `"mlp"` (défaut) | deux MLP appris conjointement | le bras doit **découvrir** la base ET l'opérateur à la fois |
| `"identity"` | `nn.Identity()`, requiert $2r=d_z$ | le bras est épinglé à la base identité |
| `"frozen"` | $\psi=T$, $\pi=T^{-1}$, **fixes, sans gradient** | la base est **donnée**, seul l'opérateur s'apprend |

Et `radius`, orthogonal à `lift`, a un troisième niveau au-delà de `"free"` : `"bounded"`
reparamétrise $\alpha := \log\sigma(\rho)/\Delta$ pour garantir $r=\sigma(\rho)\in(0,1)$ sans
condition — la démonstration complète, avec les 400 001 tirages qui la vérifient, est au
notebook 01 §3.4 ; ce notebook l'utilise telle quelle dans les cellules de budget de la §4.

### 3.1 Rappel — la fermeture close du bloc, avant de vérifier la parité

Le bloc $\bar A = \exp(\Delta(\alpha I + \omega J)) = e^{\Delta\alpha}R(\Delta\omega)$ et son
intégrale d'entrée $\Phi=\int_0^\Delta e^{sG}\,ds$ ont été dérivés au notebook 00 §2 ; la
cellule ci-dessous ne les redérive pas, elle compare la fermeture close enseignée là-bas à
`rotation_scaling_zoh` de `src/sld/predictors/blocks2x2.py` — la primitive que
`StructuredPredictor._operator_from_context` appelle à chaque pas.

In [ ]:
# --- [PARITE PACKAGE] Fermeture close (notebook 00, §2) contre blocks2x2.rotation_scaling_zoh
from sld.predictors.blocks2x2 import rotation_scaling_zoh, block_diagonal

I2 = np.eye(2)
J2 = np.array([[0.0, -1.0], [1.0, 0.0]])
DELTA = 0.05
R_BLOCKS, D_Z = 2, 4          # meme configuration que le bras oracle de la §4 (r=2, d_z=4)
TWO_R = 2 * R_BLOCKS


def expm_series(M, n_terms=100):
    '''exp(M) par serie de Taylor tronquee -- reference independante pour §3.2.'''
    S, Tm = np.eye(M.shape[0]), np.eye(M.shape[0])
    for k in range(1, n_terms + 1):
        Tm = Tm @ M / k
        S = S + Tm
    return S


def A_bar_closed(alpha, omega, dt=DELTA):
    '''exp(dt (alpha I + omega J)), forme close (notebook 00, §2.2).'''
    return np.exp(dt * alpha) * (np.cos(dt * omega) * I2 + np.sin(dt * omega) * J2)


def Phi_closed(alpha, omega, dt=DELTA):
    '''Integrale ZOH, forme close brute (valide hors de (alpha,omega)=(0,0), notebook 00 §2.5).'''
    n2 = alpha**2 + omega**2
    E, co, si = np.exp(dt * alpha), np.cos(dt * omega), np.sin(dt * omega)
    coef_I = alpha * (E * co - 1.0) + omega * E * si
    coef_J = alpha * E * si - omega * (E * co - 1.0)
    return (coef_I * I2 + coef_J * J2) / n2


rng_zoh = np.random.default_rng(7)
alpha_z = rng_zoh.uniform(-1.0, 0.5, size=R_BLOCKS)
omega_z = rng_zoh.uniform(0.1, 2.0, size=R_BLOCKS)
D_U = 2
B_cont = rng_zoh.standard_normal((TWO_R, D_U))

A_hand = np.zeros((TWO_R, TWO_R))
B_hand = np.zeros((TWO_R, D_U))
for i in range(R_BLOCKS):
    A_hand[2 * i : 2 * i + 2, 2 * i : 2 * i + 2] = A_bar_closed(alpha_z[i], omega_z[i])
    B_hand[2 * i : 2 * i + 2] = Phi_closed(alpha_z[i], omega_z[i]) @ B_cont[2 * i : 2 * i + 2]

A_blocks_pkg, B_pkg = rotation_scaling_zoh(
    torch.tensor(alpha_z), torch.tensor(omega_z), DELTA, torch.tensor(B_cont)
)
A_pkg = block_diagonal(A_blocks_pkg).numpy()
B_pkg = B_pkg.numpy()

diff_A = np.abs(A_hand - A_pkg).max()
diff_B = np.abs(B_hand - B_pkg).max()
print(f"|A_bar main - A_bar package|_inf = {diff_A:.3e}   (tolerance 1e-12)")
print(f"|B_bar main - B_bar package|_inf = {diff_B:.3e}   (tolerance 1e-12)")
assert diff_A < 1e-12 and diff_B < 1e-12

**Lecture.** Écart nul à l'arrondi machine près, en `float64` : `rotation_scaling_zoh`
calcule exactement la fermeture close enseignée au notebook 00, pas une approximation ni une
variante. La branche « série » de `rotation_scaling_zoh` (utilisée près de
$(\alpha,\omega)=(0,0)$, où la forme brute ci-dessus devient numériquement instable —
notebook 00 §2.5) n'est pas exercée ici : les tirages sont loin de la singularité par
construction, ce qui isole la comparaison sur le seul chemin de fermeture close.

### 3.2 Théorème de jauge ★

**Énoncé.** Soit $\mathcal{G} = \big(\mathbb{R}_{>0}\times SO(2)\big)^r$, agissant sur les
paramètres $(\psi,\bar A,\bar B,\pi)$ par

$$g\cdot(\psi,\bar A,\bar B,\pi) = \big(T\psi,\; T\bar A T^{-1},\; T\bar B,\; \pi\circ
T^{-1}\big), \qquad T = \mathrm{blockdiag}(c_1 R(\theta_1), \ldots, c_r R(\theta_r)).$$

Alors, pour toute condition initiale $z_0$ et toute séquence d'actions $(u_t)$, la sortie
$(\hat z_t)_{t\ge0}$ est **exactement inchangée**. De plus $T\bar A T^{-1}=\bar A$ : la
transition elle-même est un point fixe de l'action.

*Démonstration, en trois pas.*

**(i) $T$ commute avec $\bar A$.** Les deux matrices sont bloc-diagonales de même
découpage ; il suffit donc de le vérifier bloc par bloc. Sur le bloc $i$,

$$\big(c_iR(\theta_i)\big)\big(e^{\Delta\alpha_i}R(\Delta\omega_i)\big)\big(c_iR(\theta_i)\big)^{-1}
= e^{\Delta\alpha_i}R(\theta_i)R(\Delta\omega_i)R(-\theta_i)
= e^{\Delta\alpha_i}R(\theta_i+\Delta\omega_i-\theta_i) = \bar A_i,$$

où l'on a utilisé (a) que le scalaire $c_i$ se simplifie car il commute avec tout, et (b) que
$SO(2)$ est **abélien** : $R(a)R(b)=R(a+b)$. C'est le cœur de la démonstration, et c'est ce
qui rend cette jauge propre à la paramétrisation rotation-scaling : elle vient de ce que le
groupe de rotations planes commute avec lui-même. *(Le même argument vaut pour $\Phi_i$, qui
appartient à la même algèbre $\mathcal{A}=\{aI+bJ\}$ — donc $T\bar B$ se réalise aussi bien en
transformant $B$ seul, sans toucher à $\Phi$.)*

**(ii) L'état caché est simplement transporté.** Posons $h'_t$ l'état de la cellule
transformée. Par récurrence : $h'_0=T\psi(z_0)=Th_0$ ; et si $h'_t=Th_t$, alors

$$h'_{t+1} = (T\bar AT^{-1})h'_t + T\bar Bu_t = \bar A\,Th_t + T\bar Bu_t
= T\big(\bar Ah_t+\bar Bu_t\big) = T\,h_{t+1}.$$

**(iii) La sortie est invariante.** $\hat z'_t = (\pi\circ T^{-1})(h'_t) = \pi(T^{-1}Th_t) =
\pi(h_t) = \hat z_t$. $\square$

**Réalisation dans les paramètres d'un MLP.** L'action n'est pas abstraite : $T\psi$
s'obtient en remplaçant la **dernière couche linéaire** de $\psi$, $(W_L,b_L)$, par
$(TW_L,Tb_L)$ ; et $\pi\circ T^{-1}$ en remplaçant la **première couche linéaire** de $\pi$,
$W_1$, par $W_1T^{-1}$. C'est exact pour tout MLP dont les couches de bord sont affines —
donc pour l'implémentation standard. Aucune approximation n'intervient, quelle que soit la
non-linéarité interne.

### Combien de directions plates

Le groupe $\mathcal{G}$ est de dimension $2r$ : un facteur d'échelle et un angle par bloc.
L'action laisse la perte **exactement** invariante — pas approximativement, pas au voisinage
d'un optimum, mais partout et pour toute perte qui ne dépend des paramètres qu'à travers
$(\hat z_t)$. Donc :

> **[DÉMONTRÉ]** La perte possède au moins $2r$ directions de courbure exactement nulle en
> tout point où l'action est localement libre. Sur la configuration reprise plus haut
> ($r=2$) : **4 directions de dégénérescence exacte**.

Les générateurs infinitésimaux sont explicites. Pour le bloc $i$, en notant $E_i$ le
projecteur sur le bloc $i$ et $L_i=E_iJ$ :

| | direction sur $\psi$ | sur $\bar B$ | sur $\pi$ (cas linéaire $\pi=Q$) |
|---|---|---|---|
| échelle $i$ | $E_i\psi$ | $E_i\bar B$ | $-QE_i$ |
| rotation $i$ | $L_i\psi$ | $L_i\bar B$ | $-QL_i$ |

**Pourquoi c'est un problème, et pourquoi ce n'en est peut-être pas un.** Côté problème : une
variété d'optima de dimension $2r$ signifie que la norme des paramètres n'est pas identifiée,
donc que toute régularisation en norme ($\ell_2$, weight decay) agit sur la **jauge** et non
sur la fonction ; et que deux runs peuvent aboutir à des paramètres arbitrairement éloignés
tout en calculant la même chose — ce qui rend illégitime toute comparaison de paramètres entre
seeds. Côté « peut-être pas » : les réseaux profonds sont massivement invariants
(permutations, rescaling ReLU) et s'entraînent quand même. **Une dégénérescence exacte n'est
pas une preuve de difficulté d'optimisation.** C'est un fait de structure ; son effet est une
question empirique distincte.

Vérifions les deux jauges, la commutation, et le compte de directions.

In [ ]:
# --- [DEMONTRE -> verifie] Les deux jauges, sur une cellule complete (MLP + actions) --------
def mlp_init(dims, seed):
    r = np.random.default_rng(seed)
    Ws = [r.standard_normal((dims[i + 1], dims[i])) / np.sqrt(dims[i]) for i in range(len(dims) - 1)]
    bs = [0.1 * r.standard_normal(dims[i + 1]) for i in range(len(dims) - 1)]
    return [Ws, bs]


def mlp_apply(net, x):
    Ws, bs = net
    h = x
    for i, (W, b) in enumerate(zip(Ws, bs)):
        h = h @ W.T + b
        if i < len(Ws) - 1:
            h = np.tanh(h)                # non-linearite interne : la jauge n'en depend pas
    return h


def cell_rollout(psi, pi, Ab, Bb, z0, U):
    '''h0 = psi(z0) ; h_{t+1} = Ab h_t + Bb u_t ; z_hat_t = pi(h_t).'''
    h = mlp_apply(psi, z0)
    outs = []
    for t in range(U.shape[1]):
        h = h @ Ab.T + U[:, t] @ Bb.T
        outs.append(mlp_apply(pi, h))
    return np.stack(outs, axis=1)


def gauge_T(scales, angles):
    '''T = blockdiag(c_i R(theta_i)).'''
    Tm = np.zeros((TWO_R, TWO_R))
    for i in range(R_BLOCKS):
        th = angles[i]
        Tm[2 * i : 2 * i + 2, 2 * i : 2 * i + 2] = scales[i] * np.array(
            [[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]]
        )
    return Tm


def apply_gauge(psi, pi, Bb, Tm):
    '''(T psi, T Bb, pi o T^-1) realise dans les poids : derniere couche de psi, premiere de pi.'''
    Ti = np.linalg.inv(Tm)
    psi_g = [[*psi[0][:-1], Tm @ psi[0][-1]], [*psi[1][:-1], Tm @ psi[1][-1]]]
    pi_g = [[pi[0][0] @ Ti, *pi[0][1:]], [*pi[1]]]
    return psi_g, pi_g, Tm @ Bb


rg = np.random.default_rng(5)
psi = mlp_init([D_Z, 16, 16, TWO_R], 1)
pi = mlp_init([TWO_R, 16, 16, D_Z], 2)
alpha_g, omega_g = np.array([-0.20, 0.10]), np.array([2.0, -1.3])
Ab_g = np.zeros((TWO_R, TWO_R))
Phi_g = np.zeros((TWO_R, TWO_R))
for i in range(R_BLOCKS):
    Ab_g[2 * i : 2 * i + 2, 2 * i : 2 * i + 2] = A_bar_closed(alpha_g[i], omega_g[i])
    Phi_g[2 * i : 2 * i + 2, 2 * i : 2 * i + 2] = Phi_closed(alpha_g[i], omega_g[i])
B_raw = rg.standard_normal((TWO_R, D_U)) / 2
Bb_g = Phi_g @ B_raw

z0_g = rg.standard_normal((32, D_Z))
U_g = rg.standard_normal((32, 20, D_U))
ref = cell_rollout(psi, pi, Ab_g, Bb_g, z0_g, U_g)
scale_ref = np.abs(ref).max()

print("(i) commutation, bloc par bloc")
for name, Tm in [
    ("echelle pure  ", gauge_T([0.1, 7.3], [0.0, 0.0])),
    ("rotation pure ", gauge_T([1.0, 1.0], [0.9, -2.4])),
    ("echelle+rot.  ", gauge_T([0.1, 7.3], [0.9, -2.4])),
]:
    print(
        f"   {name} : |T Abar T^-1 - Abar|_inf = "
        f"{np.abs(Tm @ Ab_g @ np.linalg.inv(Tm) - Ab_g).max():.2e}"
        f"   |T Phi - Phi T|_inf = {np.abs(Tm @ Phi_g - Phi_g @ Tm).max():.2e}"
    )

print("\n(ii)+(iii) invariance EXACTE de la sortie latente (erreur relative au max de |z_hat|)")
for name, Tm in [
    ("echelle (c = 0.1, 7.3)   ", gauge_T([0.1, 7.3], [0.0, 0.0])),
    ("echelle (c = 1000, 1e-3) ", gauge_T([1000.0, 1e-3], [0.0, 0.0])),
    ("rotation (0.9, -2.4 rad) ", gauge_T([1.0, 1.0], [0.9, -2.4])),
    ("combinee                 ", gauge_T([0.1, 7.3], [0.9, -2.4])),
]:
    psi_g, pi_g, Bb_gg = apply_gauge(psi, pi, Bb_g, Tm)
    out = cell_rollout(psi_g, pi_g, Ab_g, Bb_gg, z0_g, U_g)
    print(
        f"   {name} : max|dz_hat| = {np.abs(out - ref).max():.2e}"
        f"   relatif = {np.abs(out - ref).max() / scale_ref:.2e}"
    )

**Lecture.** Deux faits, tous deux vérifiés à la précision machine. **La commutation
est exacte** : $T\bar AT^{-1}=\bar A$ à $10^{-16}$ près, pour l'échelle seule, la rotation
seule et leur combinaison — et $\Phi$ commute de la même façon avec $T$, ce qui garantit que
la jauge se réalise sur $\bar B$ sans toucher à l'intégrale ZOH, donc sans toucher à la tête
`theta_0 + W_theta c`. **La sortie latente est invariante à la précision machine**, sur un
rollout de 20 pas avec actions, à travers deux MLP à tanh, y compris pour des facteurs
d'échelle extrêmes ($10^3$ et $10^{-3}$) : l'erreur résiduelle relative est de l'ordre de
$10^{-16}$, de l'arrondi, pas une approximation.

In [ ]:
# --- [DEMONTRE -> verifie] Le compte : 2r directions plates, et elles sont independantes ----
P_lin = rg.standard_normal((TWO_R, D_Z)) / 2
Q_lin = rg.standard_normal((D_Z, TWO_R)) / 2
M_ref = Q_lin @ Ab_g @ P_lin

gens = []
for i in range(R_BLOCKS):
    E = np.zeros((TWO_R, TWO_R))
    E[2 * i : 2 * i + 2, 2 * i : 2 * i + 2] = np.eye(2)
    L = np.zeros((TWO_R, TWO_R))
    L[2 * i : 2 * i + 2, 2 * i : 2 * i + 2] = J2
    gens.append((f"echelle bloc {i}", E))
    gens.append((f"rotation bloc {i}", L))

print("derivee de M = Q Abar P le long de chaque generateur, et invariance sur un chemin fini :")
tangents = []
for name, Xg in gens:
    dM = Q_lin @ (Ab_g @ Xg - Xg @ Ab_g) @ P_lin
    Texp = expm_series(0.7 * Xg)
    M_path = (Q_lin @ np.linalg.inv(Texp)) @ Ab_g @ (Texp @ P_lin)
    tangents.append(np.concatenate([(Xg @ P_lin).ravel(), (-Q_lin @ Xg).ravel()]))
    print(
        f"   {name:>18} : |dM/de|_inf = {np.abs(dM).max():.2e}"
        f"    |M(e=0.7) - M(0)|_inf = {np.abs(M_path - M_ref).max():.2e}"
    )

Tg = np.array(tangents)
sv = np.linalg.svd(Tg, compute_uv=False)
print(f"\nvaleurs singulieres des {len(gens)} vecteurs tangents : {np.round(sv, 4)}")
print(f"rang numerique (seuil 1e-8) = {int(np.sum(sv > 1e-8))}   attendu 2r = {2 * R_BLOCKS}")
print(f"dimension de l'espace de parametres du coeur lineaire = {P_lin.size + Q_lin.size}")
assert int(np.sum(sv > 1e-8)) == 2 * R_BLOCKS

**Lecture.** Les $2r=4$ générateurs annulent exactement la dérivée de l'opérateur
prédit, l'invariance tient aussi sur un chemin **fini** ($\varepsilon=0.7$, loin de
l'infinitésimal), et les 4 vecteurs tangents sont linéairement indépendants (4 valeurs
singulières franchement non nulles). La variété plate est bien de dimension $2r=4$ dans un
espace de paramètres de dimension 32 pour le seul cœur linéaire.

**Ce qu'on ne peut pas conclure.** (i) Que ces $2r$ directions soient les **seules** : le
théorème donne une borne inférieure. (ii) Que la perte réelle soit invariante — elle l'est
*si et seulement si* elle ne dépend des paramètres qu'à travers $(\hat z_t)$ ; un terme de
régularisation en norme des poids briserait l'invariance, et c'est précisément pour cela
qu'un weight decay agirait ici sur la jauge plutôt que sur la fonction. (iii) Que cela
ralentisse l'entraînement — rien dans cette cellule ne le mesure.

### $\pi\circ\psi\neq\mathrm{Id}$, et le correctif analytique

La jauge n'est pas le seul effet de l'absence de contrainte entre $\psi$ et $\pi$. Il y a
plus élémentaire : **rien n'oblige $\pi$ à défaire $\psi$**. Deux MLP initialisés
indépendamment donnent $\pi(\psi(z))\neq z$, et la perte ne contient aucun terme qui pousse
vers l'égalité.

Pourquoi cela devrait déranger. Le rôle prêté à $\psi$ dans le récit Koopman est celui d'un
**plongement d'observables** : on monte dans un espace où la dynamique devient linéaire, on
y avance, on redescend. Si la descente n'est pas l'inverse de la montée, l'objet appris
n'est pas « la dynamique dans les coordonnées de Koopman » — c'est un réseau à trois couches
dont celle du milieu se trouve être bloc-diagonale.

**Le correctif, dans le cas linéaire.** Si $\psi$ est linéaire de matrice $P\in
\mathbb{R}^{2r\times d_z}$, poser $\pi=\psi^+$, c'est-à-dire $Q=P^+$ (pseudo-inverse de
Moore–Penrose), donne

$$QP = P^+P = \mathrm{Id}_{d_z} \iff \mathrm{rang}(P)=d_z \impliedby 2r\ge d_z\ \text{et}\
P\ \text{de rang plein}.$$

*Démonstration.* $P^+P$ est le projecteur orthogonal sur $\mathrm{ligne}(P)\subseteq
\mathbb{R}^{d_z}$ (propriété caractéristique de Moore–Penrose). Ce projecteur vaut
$\mathrm{Id}_{d_z}$ si et seulement si $\mathrm{ligne}(P)=\mathbb{R}^{d_z}$, c'est-à-dire
$\mathrm{rang}(P)=d_z$. Comme $\mathrm{rang}(P)\le\min(2r,d_z)$, la condition $2r\ge d_z$ est
**nécessaire** ; elle est suffisante pour $P$ générique. $\square$

**La littérature impose cette contrainte, et le dépôt l'omet.** Dans les auto-encodeurs de
Koopman profonds, le terme de reconstruction $\|\pi(\psi(z))-z\|^2$ est présent **par
construction** — voir [Lusch, Kutz & Brunton (2018), *Deep learning for universal linear
embeddings of nonlinear dynamics*](https://www.nature.com/articles/s41467-018-07210-0),
*Nature Communications* 9, 4950, dont la perte comporte explicitement une composante de
reconstruction en plus de la composante de prédiction. Ce n'est pas un raffinement
esthétique : c'est ce qui fait de $\psi$ un plongement plutôt qu'une couche.

In [ ]:
# --- [DEMONTRE -> verifie] pi o psi n'est pas l'identite ; le correctif, et ce qu'il ne corrige pas
zz = rg.standard_normal((256, D_Z))

print("(a) deux MLP independants : pi(psi(z)) vs z")
err_mlp = np.abs(mlp_apply(pi, mlp_apply(psi, zz)) - zz).max()
print(f"    max |pi(psi(z)) - z| = {err_mlp:.3e}   (a comparer a max|z| = {np.abs(zz).max():.2f})")

print("\n(b) coeur lineaire, deux matrices independantes")
print(f"    max |Q P - I| = {np.abs(Q_lin @ P_lin - np.eye(D_Z)).max():.3e}")

print("\n(c) correctif pi = psi^+ (pseudo-inverse), selon la relation entre 2r et d_z")
print(f"{'2r':>5} {'d_z':>5} | {'rang(P)':>8} | {'|P^+ P - I_dz|':>16} | {'|P P^+ - I_2r|':>16}")
print("-" * 62)
for two_r in [2, 3, 4, 6, 8]:
    Pm = np.random.default_rng(40 + two_r).standard_normal((two_r, D_Z))
    Qm = np.linalg.pinv(Pm)
    e_left = np.abs(Qm @ Pm - np.eye(D_Z)).max()
    e_right = np.abs(Pm @ Qm - np.eye(two_r)).max()
    print(f"{two_r:>5} {D_Z:>5} | {np.linalg.matrix_rank(Pm):>8} | {e_left:16.3e} | {e_right:16.3e}")

print("\n(d) cas degenere : P de rang deficient bien que 2r >= d_z")
Pd = np.random.default_rng(9).standard_normal((6, D_Z))
Pd[:, 3] = Pd[:, 0] + Pd[:, 1]                     # colonne dependante -> rang 3 < d_z = 4
print(
    f"    rang(P) = {np.linalg.matrix_rank(Pd)} < d_z = {D_Z}"
    f"  ->  |P^+ P - I| = {np.abs(np.linalg.pinv(Pd) @ Pd - np.eye(D_Z)).max():.3e}"
)
print("    la condition 2r >= d_z est NECESSAIRE, elle n'est pas SUFFISANTE.")

print("\n(e) la jauge d'echelle survit-elle au correctif ? (cP)^+ = P^+ / c")
for c in [0.5, 2.0]:
    Pc = c * P_lin
    print(
        f"    P -> {c}*P  :  |pinv(cP)(cP) - I| = "
        f"{np.abs(np.linalg.pinv(Pc) @ Pc - np.eye(D_Z)).max():.2e}"
        f"   |pinv(cP) - pinv(P)/{c}| = {np.abs(np.linalg.pinv(Pc) - np.linalg.pinv(P_lin) / c).max():.2e}"
    )

print("\n(f) la jauge de rotation survit-elle au correctif ? (RP)^+ = P^+ R^T")
print("    [corrige le 2026-08-19, FINDINGS.md §6 -- non teste dans une version anterieure]")
R_rot = gauge_T([1.0, 1.0], [0.9, -2.4])           # rotation pure, pas d'echelle
Pr = R_rot @ P_lin
lhs = np.linalg.pinv(Pr)
rhs = np.linalg.pinv(P_lin) @ R_rot.T
print(f"    |pinv(RP) - pinv(P) R^T|   = {np.abs(lhs - rhs).max():.2e}")
print(f"    |pinv(RP)(RP) - I|         = {np.abs(lhs @ Pr - np.eye(D_Z)).max():.2e}   (l'aller-retour reste corrige)")
assert np.abs(lhs - rhs).max() < 1e-10

**Lecture.** (a) et (b) : sans contrainte, l'aller-retour $z\to h\to z$ est faux d'un
ordre de grandeur comparable au signal lui-même — pas une petite erreur, une transformation
arbitraire. (c) : le correctif $\pi=\psi^+$ donne $P^+P=\mathrm{Id}$ à $10^{-15}$ **dès que
$2r\ge d_z$**, et échoue proprement en-deçà — la ligne $2r=2$ affiche une erreur d'ordre 1, le
comportement attendu d'une projection sur un sous-espace strict ; (d) est le garde-fou
honnête, $2r\ge d_z$ est **nécessaire et non suffisante** face à un $P$ de rang déficient.

**(e) et (f) : le correctif ne supprime ni l'une ni l'autre jauge — corrigé le 2026-08-19.**
Une version antérieure de ce paragraphe affirmait que la contrainte $\pi=\psi^+$ supprimait
la jauge de rotation ; ce n'était pas mesuré, et le calcul de (f) le contredit directement
(`FINDINGS.md` §6). Les deux identités sont exactes, à la précision machine : $(cP)^+=P^+/c$
pour l'échelle (e), $(RP)^+=P^+R^\top$ pour la rotation (f), parce que $R$ est orthogonale —
$R^\top=R^{-1}$ — et que le pseudo-inverse de Moore–Penrose commute avec une transformation
orthogonale de cette façon. Dans les deux cas le couple transformé $(cP,(cP)^+)$, resp.
$(RP,(RP)^+)$, reste dans l'orbite de la jauge : la contrainte lie $\pi$ à $\psi$, elle ne
fixe ni l'échelle ni la phase de $\psi$ lui-même. Pour supprimer la jauge d'échelle il
faudrait normaliser $P$ (par exemple $\|P\|_F=1$) ; pour supprimer la jauge de rotation il
faudrait fixer une phase par bloc. **Le pseudo-inverse répare l'aller-retour, il ne touche à
aucune des deux jauges.**

### Statut épistémique, noir sur blanc

| énoncé | statut | preuve |
|---|---|---|
| Il existe une orbite de dimension $2r$ le long de laquelle la prédiction latente est exactement constante | **DÉMONTRÉ**, vérifié à $10^{-16}$ | §3.2 |
| $\pi\circ\psi\neq\mathrm{Id}$ sans terme de reconstruction | **DÉMONTRÉ**, vérifié | ci-dessus |
| $\pi=\psi^+$ restaure l'aller-retour dès $2r\ge d_z$, et ne supprime **ni** la jauge d'échelle **ni** la jauge de rotation | **DÉMONTRÉ**, vérifié (corrigé 2026-08-19) | ci-dessus, `FINDINGS.md` §6 |
| Cette jauge **explique** un écart d'exactitude mesuré ailleurs dans le dépôt | **HYPOTHÈSE NON MESURÉE** | aucune |
| Ajouter un terme de reconstruction **améliorerait** les runs | **HYPOTHÈSE NON MESURÉE** | aucune |

Les deux dernières lignes ne sont pas des litotes. La règle appliquée ici, comme partout dans
cette série : *une dégénérescence démontrée n'autorise pas une conclusion sur l'optimisation.*
Le lien est testable — il suffit d'un bras avec terme de reconstruction, à budget et backbone
appariés — et il n'a pas été testé ici. Tant qu'il ne l'est pas, la seule chose publiable est
le théorème.

## 4. Le budget : comparer à paramètres égaux

`match_budget` (`src/sld/training/budget.py`) est le mécanisme qui rend l'équité de la §1
opposable plutôt que déclarative : il construit-et-compte — construit un modèle à un réglage
de largeur donné, compte ses paramètres entraînables, ajuste — jusqu'à tomber à $\pm5\%$ d'un
`target_params` commun à toutes les familles comparées. Pour les bras structurés (`mlp_k`,
`transformer_k`, `ssm_k`), la recherche se fait en **deux temps, identiquement pour les
trois** : un réglage grossier de `width` (le producteur de contexte, dernier palier
au-dessous de la cible) puis un réglage fin de `psi_hidden`/`pi_hidden` (~26 paramètres par
unité, résolution ~0.3%). La raison des deux temps est mesurée, pas stylistique :
`PROTOCOL.md` §13.7 montre qu'à `target=10000`, `d_z=4`, `d_u=1`, `r=4`, `transformer_k` n'a
**aucun palier** à l'intérieur du contrat sur le seul réglage grossier (`width=18` donne
$-5.58\%$, `width=20` donne $+14.48\%$ — le contexte d'attention coûte $\sim
24\,\text{width}^2$).

### 4.1 Le système conducteur, retrouvé : deux masses couplées à $c=0.10$

Rappel d'une phrase (notebook 00, §3) : `CoupledOscillators` est le système linéaire de
référence du dépôt — deux masses amorties couplées par un ressort, force sur la masse 1
seulement — dont la matrice d'état $A$ se lit directement sur les constantes physiques. C'est
ce système, à l'amortissement $c=0.10$, qui porte le bras oracle de la §4.2 : sa matrice $T$
block-diagonalise exactement $A$ à cet amortissement (`PROTOCOL.md` §28.2).

In [ ]:
# --- [PARITE PACKAGE] La matrice d'etat ecrite a la main contre CoupledOscillators.A_mat ----
from sld.systems.coupled_oscillators import CoupledOscillators

m1 = m2 = 1.0
k1, k_c, k2, c_damp = 1.0, 0.5, 1.5, 0.10        # valeurs par defaut du systeme, c = 0.10

A_hand = np.array(
    [
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
        [-(k1 + k_c) / m1, k_c / m1, -c_damp / m1, 0.0],
        [k_c / m2, -(k2 + k_c) / m2, 0.0, -c_damp / m2],
    ]
)

sysA = CoupledOscillators(c=0.10).A_mat
diff = np.abs(A_hand - sysA).max()
print(f"|A ecrite a la main - CoupledOscillators(c=0.10).A_mat|_inf = {diff:.3e}   (tolerance 1e-12)")
assert diff < 1e-12
print(A_hand)

**Lecture.** Écart nul : la classe implémente exactement la matrice écrite au notebook
00 à partir des équations du mouvement, aucune convention de signe ni de mise à l'échelle
n'a bougé entre les deux. C'est cette matrice, telle quelle, que $T$ (§4.2) block-diagonalise
à $c=0.10$.

### 4.2 Pourquoi la matcher ne suffit pas, et pourquoi la suspendre par nom

Quand `lift="identity"` ou `lift="frozen"`, $\psi$ et $\pi$ ne portent **aucun** paramètre
entraînable — les deux réglages que `match_budget` cherche perdent leur effet, et un
chercheur naïf laisserait le matcher élargir le backbone pour compenser. C'est exactement la
faute que la §1 a nommée : mesurer la largeur en l'appelant structure. Le code s'en protège
en suspendant l'appariement **par nom** plutôt qu'en élargissant la tolérance — le commentaire
de `budget.py`, cité mot pour mot :

> *"the +/-5% contract is a statement about a matched budget; the nolift arm (23.2) and
> 28.2's frozen-basis oracle arm are both declared NOT matched, so they are exempt BY NAME
> rather than by a widened tolerance, and their realised counts are returned for
> publication."*

Concrètement : un bras `lift="frozen"` **hérite** le `width` et le `psi_hidden` du bras
`on-learned` correspondant, à l'identique, plutôt que de laisser la recherche lui en
attribuer un nouveau. La cellule suivante recalcule les trois comptes et vérifie qu'ils
tombent exactement sur ceux publiés par `PROTOCOL.md` §28.2.

In [ ]:
# --- [PARITE PACKAGE] Les trois comptes de budget de PROTOCOL.md §28.2, recalcules ----------
from sld.training.budget import match_budget, count_params
from sld.predictors.structured import MLPKPredictor
from sld.predictors.mlp import MLPPredictor

cfg = yaml.safe_load((REPO / "src/sld/experiments/configs/basis_root.yaml").read_text())
oracle_arm = next(a for a in cfg["arms"] if a["overrides"].get("lift") == "frozen")
T_literal = oracle_arm["overrides"]["psi_matrix"]

kw_learned = match_budget("mlp_k", 4, 1, 10_000, delta=0.05, r=2, radius="bounded")
kw_oracle = match_budget(
    "mlp_k", 4, 1, 10_000, delta=0.05, r=2, radius="bounded",
    lift="frozen", psi_matrix=T_literal,
)
kw_off = match_budget("mlp", 4, 1, 10_000)

model_learned = MLPKPredictor(**kw_learned)
model_oracle = MLPKPredictor(**kw_oracle)
model_off = MLPPredictor(**kw_off)

total_learned = sum(p.numel() for p in model_learned.parameters())
trainable_learned = count_params(model_learned)
total_oracle = sum(p.numel() for p in model_oracle.parameters())
trainable_oracle = count_params(model_oracle)
total_off = sum(p.numel() for p in model_off.parameters())
trainable_off = count_params(model_off)

print(f"{'bras':<24} {'width':>6} {'psi_hidden':>12} | {'total':>7} {'entrainable':>12}")
print("-" * 70)
print(f"{'off (mlp)':<24} {'-':>6} {'-':>12} | {total_off:7d} {trainable_off:12d}")
print(
    f"{'on-learned (mlp_k)':<24} {kw_learned['width']:>6} {str(kw_learned['psi_hidden']):>12} "
    f"| {total_learned:7d} {trainable_learned:12d}"
)
print(
    f"{'on-oracle (mlp_k+T)':<24} {kw_oracle['width']:>6} {str(kw_oracle['psi_hidden']):>12} "
    f"| {total_oracle:7d} {trainable_oracle:12d}"
)

print(
    f"\nwidth/psi_hidden herites a l'identique : "
    f"{kw_oracle['width'] == kw_learned['width']} / {kw_oracle['psi_hidden'] == kw_learned['psi_hidden']}"
)
print(
    f"non-entrainable de l'oracle (T et T^-1) : {total_oracle - trainable_oracle}"
    f"  (attendu 2*d_z^2 = {2 * 4**2})"
)
print(
    f"ecart entrainable oracle vs on-learned  : {trainable_learned - trainable_oracle}"
    f"  (la paire psi/pi apprise que le gel retire)"
)

# recoupe contre PROTOCOL.md §28.2, cite verbatim
assert (total_off, trainable_off) == (10074, 10074)
assert (total_learned, trainable_learned) == (9997, 9997)
assert (total_oracle, trainable_oracle) == (8455, 8423)
assert total_oracle - trainable_oracle == 32 == 2 * 4**2
assert trainable_learned - trainable_oracle == 1574
print("\nles trois comptes recalcules tombent exactement sur PROTOCOL.md §28.2.")

**Lecture.** Les trois comptes recalculés ici tombent, à l'entier près, sur ceux
publiés par `PROTOCOL.md` §28.2 et épinglés par `tests/test_basis_root.py`. Le bras oracle
**hérite** `width=85` et `psi_hidden=(87,)` du bras appris — pas un réglage indépendant — et
porte $32=2d_z^2$ paramètres non entraînables (les entrées de $T$ et $T^{-1}$, rien
d'autre). L'écart entraînable de $1574$ entre `on-learned` et `on-oracle` **est** exactement
la paire $\psi/\pi$ que le gel retire — pas une coïncidence, une conséquence directe de la
construction : le bras oracle est identique au bras appris moins ce que le gel enlève. C'est
cet écart, précisément, que la contrainte de la §4.2 protège de toute contamination par un
backbone élargi.

## 5. Le piège : trois refus

Le wrapper de la §3 protège trois erreurs de configuration par un refus explicite à la
construction plutôt que par un comportement silencieusement dégradé. Les trois sont
démontrées vivantes ci-dessous : on construit le bras fautif, on capture l'erreur, on lit le
message.

In [ ]:
# --- [DEMONTRE -> verifie] Trois refus de StructuredPredictor / MLPKPredictor --------------
print("(1) lift='frozen' sans psi_matrix")
try:
    MLPKPredictor(d_z=4, d_u=1, delta=0.05, r=2, lift="frozen")
    print("    PAS D'ERREUR -- inattendu")
except ValueError as e:
    print(f"    refuse : {e}")

print("\n(2) psi_matrix fourni avec lift != 'frozen'")
try:
    MLPKPredictor(d_z=4, d_u=1, delta=0.05, r=2, lift="mlp", psi_matrix=np.eye(4))
    print("    PAS D'ERREUR -- inattendu")
except ValueError as e:
    print(f"    refuse : {e}")

print("\n(3) base singuliere (cond > 1e8)")
T_singuliere = np.eye(4)
T_singuliere[3, 3] = 0.0
try:
    MLPKPredictor(d_z=4, d_u=1, delta=0.05, r=2, lift="frozen", psi_matrix=T_singuliere)
    print("    PAS D'ERREUR -- inattendu")
except ValueError as e:
    print(f"    refuse : {e}")

**Lecture.** Les trois refus protègent trois choses distinctes. **(1)** sans
`psi_matrix`, il n'existe aucune base par défaut qui ne serait pas un choix silencieux — le
changement de base est tout le contenu de ce bras (`PROTOCOL.md` §28.2), donc il doit être
fourni explicitement. **(2)** fournir `psi_matrix` sans `lift="frozen"` serait accepté et
ignoré si le code ne le refusait pas — une configuration croirait avoir gelé une base alors
qu'elle ne l'a pas fait, l'erreur la plus difficile à détecter après coup. **(3)** une base
numériquement singulière rendrait $\pi=T^{-1}$ incapable d'inverser $\psi=T$ : le bras
porterait une mauvaise spécification silencieuse au lieu d'une base donnée honnêtement — le
refus se déclenche ici sur `cond = inf`, le cas le plus net, mais la même garde intercepte
toute base au-delà de $\mathrm{cond}>10^8$. Les trois messages nomment la cause plutôt que de
se contenter d'un type d'exception générique : c'est ce qui rend le refus utilisable en
pratique, pas seulement correct.

## 6. Ce qui a été prouvé ici, et le pont

**Prouvé.** Le facteur `structure` croise des backbones — `mlp`, le **minimal causal
transformer** (attention causale, masque, cache KV démontré équivalent au recalcul complet),
et le **Mamba-style minimal selective SSM** — avec deux rôles chacun, `off` autonome et `on`
producteur de contexte $\kappa_b$ à l'intérieur d'un seul wrapper partagé $\psi/\bar A/\pi$.
`koopman_ssm` reste constructible mais n'est structurellement jamais une cellule de ce
factoriel. Le wrapper porte une dégénérescence de jauge de dimension $2r$, démontrée et
vérifiée à la précision machine ; la contrainte $\pi=\psi^+$ répare l'aller-retour $z\to
h\to z$ mais ne supprime **ni** la jauge d'échelle **ni** la jauge de rotation — un résultat
corrigé le 2026-08-19 par son propre calcul. Le budget à $\pm5\%$ rend la comparaison entre
bras opposable plutôt que déclarative, avec une exemption **par nom** — jamais par tolérance
élargie — pour les bras dont $\psi/\pi$ ne portent aucun paramètre entraînable ; les trois
comptes qui en résultent (10074, 9997, 8455/8423) recoupent exactement `PROTOCOL.md` §28.2.
Trois refus explicites protègent la configuration du wrapper contre une base absente, une
base ignorée et une base singulière.

**Lien avec `sld`.** `src/sld/predictors/{mlp,transformer,min_ssm,koopman_ssm,structured}.py`
(les cellules et le nom hérité), `src/sld/predictors/blocks2x2.py` (la fermeture ZOH),
`src/sld/predictors/registry.py` (`STRUCTURED_FAMILIES`, la matrice complète),
`src/sld/training/budget.py` (`match_budget`, `count_params`),
`src/sld/systems/coupled_oscillators.py`, `src/sld/experiments/configs/basis_root.yaml`,
`tests/test_basis_root.py`.

**La limite honnête.** Ce notebook prouve une architecture et un contrat de comparaison — il
ne prouve rien sur ce que cette comparaison, une fois exécutée, a trouvé. Il ne rejoue ni la
campagne μ_J (§1), ni la racine `results_basis` dont `basis_root.yaml` et $T$ sont extraits
(§4) : ces résultats appartiennent au notebook 04 (le chemin des campagnes) et au notebook 05
(l'expérience oracle §28 bout en bout).

**Le pont.** Le wrapper est construit, le budget est équitable, le piège est nommé. Reste à
l'entraîner et à mesurer ce qu'il apprend : la loss JEPA, l'EMA et le stop-gradient, $e_H$ et
le VPT, et les gardes numériques qui protègent la boucle — le notebook 03.